# Type A: string polytopes of flag and Schubert varieties
An executable calculation in four stages, in the same format as the cluster-polytope notebooks:
1. Choose a dominant weight and a reduced word.
2. Generate GP paths, the string cone and weight inequalities.
3. Construct the exact SageMath polytope and inspect its geometry.
4. Verify the construction and export the result.

Use **SageMath** as the kernel and Run All. Tested with SageMath 10.8.
Coordinates are unshifted string parameters in reduced-word order. A row
`[c,a1,...,aN]` means $c+\sum a_j x_j\geq0$.

Original GP paper: Gleizer–Postnikov, [Littlewood–Richardson Coefficients via
Yang–Baxter Equation](https://arxiv.org/abs/math/9909124), IMRN 2000, 741–774.
GP conventions: Cho–Fujita–Lee, arXiv:2306.11242v2, Definitions 2.1, 2.3, 2.9.
The input may be a reduced word of **any Weyl-group element**, including the
empty word.  

Complete it as a prefix of a longest word, then set the added string coordinates to zero; this takes
a face. (See Bossinger–Fourier, arXiv:1611.06504.) 
For a regular dominant weight, this describes the string polytope toric
degeneration of X(w). A weight on a wall can lower the polytope dimension.

## 1. GP paths and weight inequalities
Every path stores its orientation index, visited crossings, turns and wire sequence.

In [2]:
"""Type-A string inequalities with explicit Gleizer--Postnikov witnesses.

Input: a longest reduced word and a dominant integral weight. No inequality
table is read. Upper wires are 1..m; crossings follow word order; wires <= k
point upward; paths go L_k -> L_(k+1); a turn r -> s has sign(s-r).
Definitions: Cho--Fujita--Lee, arXiv:2306.11242v2, 2.1, 2.3 and 2.9.
The traversal follows type_b_core.rigorous_path_normals, adding witnesses.
The accompanying audit compares it to the author's original subword code.
"""
from operator import index


def iter_rigorous_paths(word):
    word = [index(i) for i in word]
    if not word or min(word) < 1:
        raise ValueError("A nonempty positive reduced word is required")
    m, N = max(word)+1, len(word)
    order = list(range(1, m+1))
    wires = {i: [] for i in order}
    crossings = []
    for t, a in enumerate(word):
        u, v = order[a-1:a+1]
        if u > v:
            raise ValueError("Word is not reduced")
        crossings.append((u, v))
        wires[u].append(t)
        wires[v].append(t)
        order[a-1:a+1] = [v, u]
    if order != list(range(m, 0, -1)) or N != m*(m-1)//2:
        raise ValueError("Word must represent the longest element")
    neighbors = {}
    for wire, nodes in wires.items():
        for j, node in enumerate(nodes):
            neighbors[node, wire, -1] = nodes[j-1] if j else -1
            neighbors[node, wire, 1] = nodes[j+1] if j+1 < len(nodes) else N
    for k in range(1, m):
        def visit(node, wire, visited, turns, used_wires, coefficients):
            if node == N:
                if wire == k+1:
                    yield dict(k=k, visited_nodes=[j+1 for j in visited],
                                      turn_nodes=[j+1 for j in turns], wires=used_wires,
                                      normal=coefficients)
                return
            if node < 0 or node in visited:
                return
            u, v = crossings[node]
            other = v if wire == u else u
            same_direction = (u <= k) == (v <= k)
            forbidden = same_direction and (wire == u if wire <= k else wire == v)
            for dest in (wire, other):
                if dest == wire and forbidden:
                    continue
                normal = coefficients[:]
                turning = dest != wire
                if turning:
                    normal[node] = 1 if wire < dest else -1
                yield from visit(neighbors[node, dest, -1 if dest <= k else 1], dest,
                      visited+[node], turns+[node] if turning else turns,
                      used_wires+[dest] if turning else used_wires, normal)
        yield from visit(wires[k][-1], k, [], [], [k], [0]*N)


def rigorous_paths(word):
    """Compatibility helper for audits that explicitly need every path witness."""
    return list(iter_rigorous_paths(word))


def reduced_word(permutation):
    """One reduced word for a one-line permutation, without enumerating words.

    Adjacent swaps act on positions, starting at the identity. Moving each
    requested entry left creates exactly one inversion per swap: O(n^2) time.
    Different reduced words can give different string polytopes; this returns
    one deterministic choice, not a canonical polytope for the permutation.
    """
    permutation = list(map(index, permutation))
    n = len(permutation)
    if n < 2 or sorted(permutation) != list(range(1,n+1)):
        raise ValueError("Enter a permutation of 1,...,n with n >= 2")
    order, word = list(range(1,n+1)), []
    for i, value in enumerate(permutation):
        j = order.index(value, i)
        while j > i:
            order[j-1], order[j] = order[j], order[j-1]
            word.append(j)
            j -= 1
    return word


def longest_word(rank):
    """The natural triangular reduced word for w0 in type A_rank."""
    rank = index(rank)
    if rank < 1:
        raise ValueError("Type A rank must be positive")
    return [j for i in range(1,rank+1) for j in range(i,0,-1)]


def weight_rows(word, weight):
    word, weight = list(map(index, word)), list(map(index, weight))
    if not weight or any(a < 0 for a in weight) or any(a < 1 or a > len(weight) for a in word):
        raise ValueError("Weight must be a dominant integral weight of the correct rank")
    rows = []
    for j, root in enumerate(word):
        normal = [0]*len(word)
        normal[j] = -1
        for t in range(j+1, len(word)):
            normal[t] = -2 if root == word[t] else 1 if abs(root-word[t]) == 1 else 0
        rows.append([weight[root-1]] + normal)
    return rows


def extend_to_longest(word, rank, reverse=False):
    """Complete a reduced prefix by right ascents, keeping its coordinates first."""
    rank = index(rank)
    if rank < 1:
        raise ValueError("Type A rank must be positive")
    order = list(range(1, rank+2))
    extended = list(map(index, word))
    for a in extended:
        if not 1 <= a <= rank:
            raise ValueError("Simple-root index outside the selected rank")
        if order[a-1] > order[a]:
            raise ValueError("The word is not reduced")
        order[a-1], order[a] = order[a], order[a-1]
    permutation = order[:]
    while any(order[j] < order[j+1] for j in range(rank)):
        choices = range(rank-1,-1,-1) if reverse else range(rank)
        j = next(j for j in choices if order[j] < order[j+1])
        order[j], order[j+1] = order[j+1], order[j]
        extended.append(j+1)
    return extended, permutation


def string_record(word, weight, reverse_completion=False, include_paths=True):
    """String polytope for X(w), including w != w0 and dominant wall weights.

    Complete the word as a prefix of w0, take the GP cone, and set all added
    string coordinates to zero (a face, NOT a projection). Then retain the
    original word's weight inequalities. The result has len(word) coordinates.
    """
    word, weight = list(map(index, word)), list(map(index, weight))
    weights = weight_rows(word, weight)
    extended, permutation = extend_to_longest(word, len(weight), reverse_completion)
    d = len(word)
    paths, normals = [], set()
    for p in iter_rigorous_paths(extended):
        normal = p["normal"][:d]
        if any(normal):
            normals.add(tuple(normal))
            if include_paths:
                paths.append(dict(p, ambient_normal=p["normal"], normal=normal))
    cone = [[0]+list(a) for a in sorted(normals)]
    return dict(word=word, weight=weight, rank=len(weight), dimension=d,
                permutation=permutation, longest=word == extended, completion_word=extended,
                zero_coordinates=list(range(d+1,len(extended)+1)), paths=paths, path_witnesses_included=include_paths,
                cone_H=cone, weight_H=weights, H=cone+weights if d else [[0]],
                coordinates="unshifted string coordinates in reduced-word order",
                H_convention="constant + normal dot x >= 0")


def string_polytope(word, weight):
    data = string_record(word, weight, include_paths=False)
    return Polyhedron(ieqs=data['H'], base_ring=QQ, ambient_dim=len(word))

## 2. Choose a word and weight
The length of λ determines the type A rank. Enter a reduced word in that rank; it need not represent the longest element. Examples: `word=[1,2]`, `weight=[1,1]` for a Schubert surface; `word=[]` for a point. For SL5 use four weight coefficients. 

In [3]:
rank = 4  # type A_rank = SL(rank+1); increase without supplying a word table
weight = [1]*rank  # rho; [2]*rank for 2rho
word = longest_word(rank)

# Alternatively choose a permutation or enter the word directly:
# permutation = [2,3,1]  # one-line notation in S_3; use rank=2 and two weights
# word = reduced_word(permutation)
# word = [1,2]
KEEP_PATH_WITNESSES = False
COMPUTE_GEOMETRY = len(word) <= 4  # set True explicitly for a larger calculation
record = string_record(word, weight, include_paths=KEEP_PATH_WITNESSES)
P = Polyhedron(ieqs=record['H'], base_ring=QQ, ambient_dim=len(word)) if COMPUTE_GEOMETRY else None
print('Word:', word, '| weight:', weight)
print('Permutation:', record['permutation'], '| longest:', record['longest'])
print('String coordinates:', len(word), '| defining cone rows:', len(record['cone_H']))
if P is not None:
    print('Dimension:', P.dim(), '| vertices:', P.n_vertices())
    print('f-vector:', P.f_vector())
else:
    print('Exact H-representation generated. Geometry is deferred; set COMPUTE_GEOMETRY=True to request it.')
print('Defining inequalities [constant, normal]:')
for row in record['H']: print(row)

Word: [1, 2, 1, 3, 2, 1, 4, 3, 2, 1] | weight: [1, 1, 1, 1]
Permutation: [5, 4, 3, 2, 1] | longest: True
String coordinates: 10 | defining cone rows: 10
Exact H-representation generated. Geometry is deferred; set COMPUTE_GEOMETRY=True to request it.
Defining inequalities [constant, normal]:
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, -1]
[0, 0, 0, 0, 0, 0, 0, 0, 1, -1, 0]
[0, 0, 0, 0, 0, 0, 0, 1, -1, 0, 0]
[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 1, -1, 0, 0, 0, 0]
[0, 0, 0, 0, 1, -1, 0, 0, 0, 0, 0]
[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 1, -1, 0, 0, 0, 0, 0, 0, 0]
[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[1, -1, 1, -2, 0, 1, -2, 0, 0, 1, -2]
[1, 0, -1, 1, 1, -2, 1, 0, 1, -2, 1]
[1, 0, 0, -1, 0, 1, -2, 0, 0, 1, -2]
[1, 0, 0, 0, -1, 1, 0, 1, -2, 1, 0]
[1, 0, 0, 0, 0, -1, 1, 0, 1, -2, 1]
[1, 0, 0, 0, 0, 0, -1, 0, 0, 1, -2]
[1, 0, 0, 0, 0, 0, 0, -1, 1, 0, 0]
[1, 0, 0, 0, 0, 0, 0, 0, -1, 1, 0]
[1, 0, 0, 0, 0, 0, 0, 0, 0, -1, 1]
[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, -1]


In [7]:
from sage.all import WeylGroup

rank = 4
weight = [1] * rank

W = WeylGroup(["A", rank])
words = sorted(W.long_element().reduced_words())

word_index = 10                   
word = list(words[word_index])

print("all reduced words:", len(words))
print("choosen word:", word)


KEEP_PATH_WITNESSES = False
COMPUTE_GEOMETRY = len(word) <= 4  # set True explicitly for a larger calculation
record = string_record(word, weight, include_paths=KEEP_PATH_WITNESSES)
P = Polyhedron(ieqs=record['H'], base_ring=QQ, ambient_dim=len(word)) if COMPUTE_GEOMETRY else None
if P is not None:
    print('Dimension:', P.dim(), '| vertices:', P.n_vertices())
    print('f-vector:', P.f_vector())
else:
    print('Exact H-representation generated. Geometry is deferred; set COMPUTE_GEOMETRY=True to request it.')
print('Defining inequalities [constant, normal]:')
for row in record['H']: print(row)

all reduced words: 768
choosen word: [1, 2, 1, 4, 3, 2, 4, 1, 3, 2]
Exact H-representation generated. Geometry is deferred; set COMPUTE_GEOMETRY=True to request it.
Defining inequalities [constant, normal]:
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, -1]
[0, 0, 0, 0, 0, 0, 0, 0, 1, 0, -1]
[0, 0, 0, 0, 0, 0, 0, 1, 0, -1, 0]
[0, 0, 0, 0, 0, 0, 1, 0, -1, 0, 0]
[0, 0, 0, 0, 0, 0, 1, 0, 0, -1, 0]
[0, 0, 0, 0, 0, 1, -1, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 1, 0, -1, 0, 0, 0]
[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 1, -1, 0, 0, 0, 0, 0, 0, 0]
[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[1, -1, 1, -2, 0, 0, 1, 0, -2, 0, 1]
[1, 0, -1, 1, 0, 1, -2, 0, 1, 1, -2]
[1, 0, 0, -1, 0, 0, 1, 0, -2, 0, 1]
[1, 0, 0, 0, -1, 1, 0, -2, 0, 1, 0]
[1, 0, 0, 0, 0, -1, 1, 1, 0, -2, 1]
[1, 0, 0, 0, 0, 0, -1, 0, 1, 1, -2]
[1, 0, 0, 0, 0, 0, 0, -1, 0, 1, 0]
[1, 0, 0, 0, 0, 0, 0, 0, -1, 0, 1]
[1, 0, 0, 0, 0, 0, 0, 0, 0, -1, 1]
[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, -1]


## References and provenance
- [Gleizer–Postnikov] Oleg Gleizer and Alexander Postnikov, Littlewood–Richardson coefficients via Yang–Baxter equation, International Mathematics Research Notices, Volume 2000, Issue 14 (2000), 741–774. (https://doi.org/10.1155/S1073792800000416)
    
Copyright © Shin Hong, Jeongsoo Kim, Eunjeong Lee.
Prepared with assistance from ChatGPT.